# Chapter 2 - Lab 3: Episodic memory - remembering the customer next week

**Episodic memory** = remembering *past experiences*. When Priya contacts NovaMart again next week,
the agent should already know that two repairs failed, a replacement was approved, she upgraded and paid
Rs 5,500, and she is a Platinum customer at risk of churning. She should not have to tell her story again.

*Finance version*: a portfolio agent recalled past rebalancing episodes (allocations and what worked).
*Customer-care version*: a care agent recalls past **support episodes** for the same customer.

Pipeline:
1. Hold a conversation (working memory, as in Lab 2).
2. Condense it into a structured **episode** (JSON) with an LLM, two ways: LangChain and the OpenAI Responses API.
3. **Embed** the episode (`text-embedding-3-small`) and store it in a vector database (**ChromaDB**).
4. In a new session, **retrieve** similar episodes and inject them into the prompt.

In [ ]:
%pip install -q openai-agents langchain-openai chromadb

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

In [ ]:
import json, os
from agents import Agent, Runner, SQLiteSession

# Step 1: the conversation (same agent and turns as Lab 2)

In [ ]:
care_agent = Agent(
    name="NovaMart Care Agent",
    instructions="""You are a senior NovaMart customer-care agent helping a customer by chat.
Your goal is to resolve the customer's problem fairly, quickly and within policy.

Policy you must apply:
- Repeat-failure rule: if the same defect persists after 2 repair attempts within warranty,
  the customer is entitled to a replacement of the same model.
- Upgrades: a customer may upgrade the replacement to a higher model by paying the price difference.
- Goodwill credits: up to Rs 1,000 at agent level, up to Rs 5,000 with supervisor approval.
  Credits can be adjusted against an upgrade payment.
- Large-appliance replacements are delivered and installed within 5-7 working days
  (priority dispatch within 72 hours for Platinum customers when stock is available).

Your role:
- Keep track of the facts the customer gives you (order, prices, dates, what was already tried).
- Do the arithmetic for the customer when money is involved; show your calculation.
- Be empathetic, concise and specific. State assumptions when information is missing.""",
    model="gpt-4o-mini",
)

In [ ]:
turn_1 = """Hi, I'm Priya Sharma, customer ID C1001. My FrostLine 340L refrigerator (order ORD-77812),
bought for Rs 62,990 and delivered on 18 September, stopped cooling on 9 October. What should I do?"""

turn_2 = """A technician already came on 12 October and replaced a sensor; it failed again two days later.
A second visit on 19 October also failed: they said a relay part is out of stock."""

turn_3 = "So what are my options now?"

turn_4 = """I'll take the replacement, but can I upgrade to the FrostLine 400L model instead? It costs Rs 71,490.
Also, your colleague promised me a Rs 3,000 goodwill credit for the food I had to throw away."""

turn_5 = "So exactly how much do I need to pay for the upgrade?"

turn_6 = "Fine. Please confirm delivery for Thursday and summarise everything we agreed."

In [ ]:
async def show_working_memory(session) -> list[tuple[str, str]]:
    """Print the session (working memory) and return it as (role, text) pairs."""
    items = await session.get_items()
    history = []
    for i, msg in enumerate(items, 1):
        role = msg.get("role")
        content = msg.get("content", "")
        if role == "assistant":
            text = content[0]["text"] if isinstance(content, list) else str(content)
        elif role == "user":
            text = content if isinstance(content, str) else str(content)
        else:
            continue
        history.append((role, text))
        print(f"  {i}. \033[95m{role}\033[0m: {text[:300]}{'...' if len(text) > 300 else ''}")
        print("--" * 40)
    return history

In [ ]:
session_week1 = SQLiteSession("priya_week_1")
for turn in [turn_1, turn_2, turn_3, turn_4, turn_5, turn_6]:
    result = await Runner.run(care_agent, turn, session=session_week1)
    print(f"\nCUSTOMER: {turn}\nAGENT: {result.final_output}\n" + "-" * 80)

In [ ]:
history = await show_working_memory(session_week1)
conversation_history_string = "\n\n".join(f'{role}: """{text}"""' for role, text in history)

# Step 2: generate the episode

In [ ]:
episodic_memory_prompt_template = """You are an episodic-memory agent supporting a principal customer-care agent.
Your job is to create a concise episodic memory of a support session:
* Summarise the conversation (conversation_summary),
* Record the case trajectory: how the issue, the options offered and the money involved evolved, with dates (case_trajectory),
* Identify what worked (what_worked) and what did not (what_didnt_work),
* Extract key points to remember next time this customer contacts us (key_points).

You receive the full conversation between the customer and the principal agent.

OUTPUT FORMAT - return only valid JSON in this exact schema:
{{
  "conversation_summary": string,   // 3-5 sentences: customer, product, problem, resolution, money
  "case_trajectory": string,        // ordered steps with dates and amounts
  "what_worked": string,            // e.g. clear policy explanation, correct arithmetic, empathy
  "what_didnt_work": string,        // e.g. repeated failed repairs, slow escalation, unclear commitments
  "key_points": ["string"]          // facts and commitments to remember for future contacts
}}

FEW-SHOT EXAMPLES
GOOD conversation_summary:
"Platinum customer reported a washing machine leaking after one month. Two repairs failed, so the agent
applied the repeat-failure rule and approved a replacement. The customer upgraded to the 8 kg model and paid
Rs 4,200 after a Rs 1,000 goodwill credit. Delivery confirmed for 14 March."
BAD conversation_summary:
"The customer had a problem. The agent helped. It was resolved." (no product, no numbers, no commitments)

GOOD what_didnt_work:
"The customer had to explain the history three times across channels; the second technician visit
was booked without the spare part, wasting a visit."
BAD what_didnt_work:
"Nothing went wrong." (unrealistic; no operational insight)

General rules:
- Be concise and factual. Do not invent facts, amounts or dates that are not in the conversation.
- Output only valid JSON and nothing else.

Here is the conversation between the customer and the principal care agent:
{conversation}
"""

## Method 1: LangChain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
generate_episodic_memory = (ChatPromptTemplate.from_template(episodic_memory_prompt_template)
                            | llm | JsonOutputParser())

episodic_memory = generate_episodic_memory.invoke({"conversation": conversation_history_string})
episodic_memory   # a Python dict

## Method 2: OpenAI Responses API (JSON mode)

In [ ]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-4.1-mini",
    input=[{"role": "user", "content": episodic_memory_prompt_template.format(conversation=conversation_history_string)}],
    text={"format": {"type": "json_object"}},
)
episodic_memory_v2 = json.loads(response.output_text)   # keep it as a dict, not a string
print(json.dumps(episodic_memory_v2, indent=2, ensure_ascii=False))

# Step 3: store the episode in ChromaDB

Each episode is stored twice:
* as **text** in `documents` (embedded, so we can search by meaning), and
* as **metadata** (structured fields, so we can filter, for example by `customer_id`).

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

embedding_function = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_API_KEY"], model_name="text-embedding-3-small")

persistent_client = chromadb.PersistentClient(path="./chroma_care_episodes")   # a folder on disk
collection = persistent_client.get_or_create_collection(
    "episodic_memory_customer_care", embedding_function=embedding_function)

In [ ]:
def add_episode(episode: dict, episode_id: str, customer_id: str) -> None:
    document_text = (
        f"Conversation Summary: {episode['conversation_summary']}\n"
        f"Case Trajectory: {episode['case_trajectory']}\n"
        f"What Worked: {episode['what_worked']}\n"
        f"What Didn't Work: {episode['what_didnt_work']}\n"
        f"Key Points: {', '.join(episode['key_points'])}"
    )
    metadata = {
        "customer_id": customer_id,
        "conversation_summary": episode["conversation_summary"],
        "case_trajectory": episode["case_trajectory"],
        "what_worked": episode["what_worked"],
        "what_didnt_work": episode["what_didnt_work"],
        "key_points": " | ".join(episode["key_points"]),   # Chroma metadata must be flat
    }
    collection.upsert(ids=[episode_id], documents=[document_text], metadatas=[metadata])

add_episode(episodic_memory, "C1001-episode-1", "C1001")
print("Episodes stored:", collection.count())

# Step 4: one week later, a new session

Priya writes again. We search her past episodes **by meaning**, filtered to her customer ID:

In [ ]:
new_message = """Hi, it's Priya again (C1001). The new refrigerator was delivered but the ice maker makes a
loud noise every few minutes. Honestly I'm losing patience with this product."""

recap = collection.query(query_texts=[new_message], n_results=3, where={"customer_id": "C1001"})
print(recap["documents"][0][0])

In [ ]:
session_week2 = SQLiteSession("priya_week_2")   # fresh working memory

message_with_memory = new_message + f"""

Relevant history from past support episodes with this customer (use it as context, do not read it back verbatim):
{recap['documents'][0][0]}

Acknowledge the history so the customer does not have to repeat herself, and propose the next best action.
"""

output = await Runner.run(care_agent, message_with_memory, session=session_week2)
print(output.final_output)

With episodic memory the agent should acknowledge the earlier failures and the upgrade, recognise a
frustrated Platinum customer, and escalate faster instead of starting the troubleshooting script from zero.

# All in: a chat loop that writes its own episode on exit

Wrapping the steps into functions:
* each user message is enriched with retrieved episodes,
* the agent answers inside a working-memory session,
* typing `exit` condenses the session into an episode and stores it in ChromaDB.

In [ ]:
def concat_working_memory(items) -> list[tuple[str, str]]:
    history = []
    for msg in items:
        role, content = msg.get("role"), msg.get("content", "")
        if role == "assistant":
            history.append((role, content[0]["text"] if isinstance(content, list) else str(content)))
        elif role == "user":
            history.append((role, content if isinstance(content, str) else str(content)))
    return history


def build_episode(history) -> dict:
    conversation = "\n\n".join(f'{role}: """{text}"""' for role, text in history)
    return generate_episodic_memory.invoke({"conversation": conversation})


def with_memory(user_message: str, customer_id: str) -> str:
    recap = collection.query(query_texts=[user_message], n_results=3, where={"customer_id": customer_id})
    episodes = "\n---\n".join(recap["documents"][0]) if recap["documents"][0] else "No past episodes."
    return user_message + f"\n\nRelevant past support episodes (context only):\n{episodes}"


async def chat_loop(customer_id: str, episode_id: str, get_input=input):
    session = SQLiteSession(episode_id)
    while True:
        user_message = get_input("\nCustomer: ")
        if user_message.strip().lower() == "exit":
            items = await session.get_items()
            if items:
                episode = build_episode(concat_working_memory(items))
                add_episode(episode, episode_id, customer_id)
                print("Episode stored:", json.dumps(episode, indent=2, ensure_ascii=False))
            break
        result = await Runner.run(care_agent, with_memory(user_message, customer_id), session=session)
        print(f"Agent: {result.final_output}")

In [ ]:
# Interactive: type messages as the customer, then "exit" to store the episode.
# Example messages:
#   "Hi, it's Priya (C1001). The technician replaced the ice-maker fan and it is quiet now. Thanks!"
#   "exit"
await chat_loop("C1001", "C1001-episode-2")

# Two more kinds of memory (concepts)

**Semantic memory** = facts and knowledge, not experiences: product manuals, return policies, warranty
terms, known-issue bulletins. It is stored in a vector database and retrieved with RAG (Chapter 10).

**Procedural memory** = *how to do things*: standard operating procedures. For a replacement request:
1. verify identity and order; 2. check warranty and repair history; 3. apply the repeat-failure rule;
4. check stock and dispatch options; 5. compute any price difference and credits; 6. confirm and log the commitment.

## Discussion
1. Which customer interactions deserve an episode, and which are noise (e.g., "thanks!")?
2. Data protection: how long should episodes be kept, and who may read them?
3. How would episodic memory change your contact centre's **repeat-contact rate** and **average handle time**?